# Exemplo: Calibração
----------------------

Este exemplo mostra como calibrar um classificador com o experionml.

Os dados utilizados são uma variação do [conjunto de dados de clima australiano](https://www.kaggle.com/jsphyg/weather-dataset-rattle-package) do Kaggle. Você pode baixá-lo [aqui](https://github.com/tvdboom/ExperionML/blob/master/examples/datasets/weatherAUS.csv). O objetivo deste conjunto é prever se vai chover amanhã treinando um classificador binário com a variável alvo `RainTomorrow`.

## Carregar os dados

In [1]:
# Import packages
import pandas as pd
from experionml import ExperionMLClassifier

In [2]:
# Carregar os dados
X = pd.read_csv("./datasets/weatherAUS.csv")

# Let's have a look
X.head()

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,MelbourneAirport,18.0,26.9,21.4,7.0,8.9,SSE,41.0,W,SSE,...,95.0,54.0,1019.5,1017.0,8.0,5.0,18.5,26.0,Yes,0
1,Adelaide,17.2,23.4,0.0,NaN,NaN,S,41.0,S,WSW,...,59.0,36.0,1015.7,1015.7,NaN,NaN,17.7,21.9,No,0
2,Cairns,18.6,24.6,7.4,3.0,6.1,SSE,54.0,SSE,SE,...,78.0,57.0,1018.7,1016.6,3.0,3.0,20.8,24.1,Yes,0
3,Portland,13.6,16.8,4.2,1.2,0.0,ESE,39.0,ESE,ESE,...,76.0,74.0,1021.4,1020.5,7.0,8.0,15.6,16.0,Yes,1
4,Walpole,16.4,19.9,0.0,NaN,NaN,SE,44.0,SE,SE,...,78.0,70.0,1019.4,1018.9,NaN,NaN,17.4,18.1,No,0


## Executar o pipeline

In [3]:
experionml = ExperionMLClassifier(X, "RainTomorrow", n_rows=1e4, verbose=1, warnings=False)

# Apply data cleaning steps
experionml.clean()
experionml.impute(strat_num="median", strat_cat="most_frequent")
experionml.encode(strategy="target", max_onehot=5, infrequent_to_value=0.05)

# Treine uma SVM linear
experionml.run("gnb")

<< ================== ExperionML ================== >>

Configuração ==================== >>
Tarefa do algoritmo: Binary classification.

Estatísticas do conjunto de dados ==================== >>
Formato: (10000, 22)
Tamanho do conjunto de train: 8000
Tamanho do conjunto de test: 2000
-------------------------------------
Memória: 1.76 MB
Escalonado: False
Valores ausentes: 22177 (10.1%)
Atributos categóricos: 5 (23.8%)
Duplicatas: 1 (0.0%)

Ajustando Cleaner...
Limpando os dados...
Ajustando Imputer...


Imputando valores ausentes...
Ajustando Encoder...


Codificando colunas categóricas...

Training ========================= >>
Models: GNB
Metric: f1


Results for GaussianNB:
Fit ---------------------------------------------
Train evaluation --> f1: 0.5705
Test evaluation --> f1: 0.5736
Time elapsed: 0.097s
-------------------------------------------------
Time: 0.097s


Resultados finais ==================== >>
Tempo total: 0.107s
-------------------------------------
GaussianNB --> f1: 0.5736


## Analisar os resultados

In [4]:
# Verifique a calibração do modelo
experionml.plot_calibration()

In [5]:
# Vamos tentar melhorá-la usando o método calibrate
experionml.winner.calibrate(method="isotonic")

Results for GaussianNB:
Fit ---------------------------------------------
Train evaluation --> f1: 0.516
Test evaluation --> f1: 0.4816
Time elapsed: 0.172s


In [6]:
# And check again...
experionml.plot_calibration()